#### Functions from Assignment 0

In [4]:
import import_ipynb
from random import randint
from math import sqrt
from sympy import factorint

In [5]:
# Finds the m- ary expansion of n
def getExpansion (n ,m):
    listOfDigits =[]
    while n >= m:
        digit =n%m
        listOfDigits . append ( digit )
        n =(n - digit ) // m
    listOfDigits.append(n)
    return listOfDigits
    
def intToText (n):
    t= getExpansion(n ,256)
    myString =''
    for i in t :
        myString = myString + chr(i)
    return myString

def textToInt (s):
    n =0
    k =0
    for i in s :
        n=n +ord( i) *(256** k)
        k=k +1
    return n

# Output the multiplicative inverse of a modulo p
def multInverse (a , p):
    result = extendedGCD (a , p)
    if result [0]!=1: # Error message if a and p are not relatively prime
        s=" Numbers needs to be relatively prime "
        return s
    inv = result [1]% p
    return inv

#extended euclidean algorithm
# Output [r,s,t] satisfying s*a+t*b=r=gcd(a,b)
def extendedGCD(a , b):
    r0 , r=a ,b
    s0 , s =1 ,0
    t0 , t =0 ,1
    while (r >0):
        tempr , temps , tempt =r ,s ,t
        q= r0 // r
        r ,s , t=r0 - q*r ,s0 -q*s ,t0 - q*t
        r0 , s0 , t0 = tempr , temps , tempt
    return [r0 ,s0 , t0 ]

def fast2Power (a ,n ,m):
    res = 1
    while n > 0:
        if n % 2 == 1: #If the bit is 1 multiply by the corresponding square
            res = ( res * a ) % m
        a =( a * a) % m
        n = n // 2
    return res

### Functions for Assignment 3

In [7]:
def findSquareRoot (N ,p) :
    N0 = N % p
    if fast2Power (N0, (p - 1) // 2 ,p) == 1: #Euler ’s criterion
        if p % 4 == 3:
            x1 = fast2Power (N0, (p + 1) // 4 , p) #see assignment 2 exercise 2 theoretical part
            y1 = p - x1
            return [x1 , y1]
        else :
            for i in range(1 , (( p - 1) // 2) + 1):
                if (i * i) % p == N0:
                    x1 = i
                    y1 = p - i
                    return [x1 , y1]
    return []

def generateCurve (E , p):
    if isElliptic (E , p) == False :
        print (" This is not an elliptic curve ")
        return None
    A, B = E
    listOfPoints =["O"]
    for x in range (p):
        a =(x**3 + A*x + B) % p
        if a == 0:
            listOfPoints.append ([x ,0])
        if fast2Power (a, (p - 1) // 2, p) == 1: # Euler ’s criterion there are solutions
            y1, y2 = findSquareRoot(a, p)
            listOfPoints.append([x, y1])
            listOfPoints.append([x, y2])
    return listOfPoints

# Part I (common part)

## Implementation part

### Exercise 1

In [11]:
def isElliptic (E ,p ):
    A=E [0]
    B=E [1]
    discr =(4*( A **3) +27*( B **2) )%p
    return discr !=0

def pointOnCurve (P ,E ,p) :
    if P == "O":
        return True
    else :
        A=E [0]
        B=E [1]
        x=P [0]
        y=P [1]
        return (y **2) %p ==( x **3+ A* x+B) %p

In [12]:
E = [0,7] #represents the elliptic curve y^2 = x^3 + 7
p = 2**256 - 2**32 - 977
x, y = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798, 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
P = [x, y]
if not isElliptic(E,p):
    print("E is not elliptic!")
elif not pointOnCurve(P, E, p):
    print("P is not on the curve!")
else:
    print("P is on the curve!")

P is on the curve!


### Exercise 2

In [14]:
E = [5,12]
p = 13
C = generateCurve(E, p)
C

['O', [0, 5], [0, 8], [2, 2], [2, 11], [7, 0], [10, 3], [10, 10]]

In [15]:
len(C) <= p + 1 + 2*sqrt(p) and len(C) >= p + 1 - 2*sqrt(p)

True

### Exercise 3

#### (a)

In [18]:
def addPoints(P,Q,E,p):
    A = E[0]
    B = E[1]
    if P == "O":
        return Q
    elif Q == "O":
        return P
    x1, x2 = P[0], Q[0]
    y1, y2 = P[1], Q[1]
    if x1 == x2 % p and y1 == -y2 % p:
        return "O"
    else:
        if P != Q:
            lmbda = (y2 - y1) * multInverse(x2 - x1, p) % p
        else:
            lmbda = (3 * fast2Power(x1, 2, p) + A) * multInverse(2*y1, p) % p
        x3 = (fast2Power(lmbda, 2, p) - x1 - x2) % p
        y3 = (lmbda * (x1 - x3) - y1) % p
        return [x3, y3]

#### (b)

In [20]:
E = [5, 12]
p = 13
P1, Q1 = "O", "O"
P2, Q2 = "O", [0,5]
P3, Q3 = [2, 2], "O"
P4, Q4 = [10, 3], [10, -3]
P5, Q5 = [2, 2], [2, 11]
P6, Q6 = [10, 10], [0, 5]
P7, Q7 = [10, 3], [7, 0]
P8, Q8 = [10, 3], [10, 3]
P9, Q9 = [0, 5], [2, 11]
Plist = [P1, P2, P3, P4, P5, P6, P7, P8, P9]
Qlist = [Q1, Q2, Q3, Q4, Q5, Q6, Q7, Q8, Q9]
for i in range(len(Plist)):
    print(i + 1, ":", Plist[i], "+", Qlist[i], "=", addPoints(Plist[i], Qlist[i], E, p))

1 : O + O = O
2 : O + [0, 5] = [0, 5]
3 : [2, 2] + O = [2, 2]
4 : [10, 3] + [10, -3] = O
5 : [2, 2] + [2, 11] = O
6 : [10, 10] + [0, 5] = [0, 8]
7 : [10, 3] + [7, 0] = [10, 10]
8 : [10, 3] + [10, 3] = [7, 0]
9 : [0, 5] + [2, 11] = [7, 0]


### Exercise 4

#### (a)

In [23]:
def doubleAndAdd(P, n, E, p):
    res = "O"
    while n > 0:
        if n % 2 == 1:
            res = addPoints(res, P, E, p)
        P = addPoints(P, P, E, p)
        n = n // 2
    return res    

#### (b)

In [25]:
E = [0,7] #represents the elliptic curve y^2 = x^3 + 7
p = 2**256 - 2**32 - 977
x, y = 0x79BE667EF9DCBBAC55A06295CE870B07029BFCDB2DCE28D959F2815B16F81798, 0x483ADA7726A3C4655DA4FBFC0E1108A8FD17B448A68554199C47D08FFB10D4B8
G = [x, y]
n = 165717357988647532
nG = doubleAndAdd(G, n, E, p)
print("nG =\n", nG)

nG =
 [51524656361346136203439460631008348936841590752868015863048885409956560359198, 79203802285035089814150287439488171030915061256999835886514121114625556531371]


### Exercise 5

### (a)

In [220]:
def addPoints(P,Q,E,N):
    A = E[0]
    B = E[1]
    if P == "O":
        return Q
    elif Q == "O":
        return P
    x1, x2 = P[0], Q[0]
    y1, y2 = P[1], Q[1]
    if x1 == x2 % N and y1 == -y2 % N:
        return "O"
    else:
        if P != Q:
            d1 = extendedGCD(x2 - x1, N)[0]
            if d1 != 1:
                return [-1, d1]
            lmbda = (y2 - y1) * multInverse(x2 - x1, N) % N
        else:
            d2 = extendedGCD(2 * y1, N)[0]
            if d2 != 1:
                return [-1, d2]
            lmbda = (3 * fast2Power(x1, 2, N) + A) * multInverse(2 * y1, N) % N
        x3 = (fast2Power(lmbda, 2, N) - x1 - x2) % N
        y3 = (lmbda * (x1 - x3) - y1) % N
        return [x3, y3]
        
def doubleAndAdd(P, n, E, p):
    res = "O"
    while n > 0:
        if n % 2 == 1:
            res = addPoints(res, P, E, p)
        P = addPoints(P, P, E, p)
        n = n // 2
    return res    

In [29]:
def factorLenstra(N, boundary = 50):
    while True:
        A = randint(0, N - 1)
        a = randint(0, N - 1)
        b = randint(0, N - 1)
        B = (fast2Power(b, 2, N) - fast2Power(a, 3, N) - A * a) % N
        if (4 * fast2Power(A, 3, N) + 27 * fast2Power(B, 2, N)) % N != 0:
            break
    E = [A, B]
    P = [a, b]
    for i in range(2, boundary):
        x1, x2 =  doubleAndAdd(P, i, E, N)
        if x1 == -1:
            return x2 
        P = [x1, x2]
    return 1  

#### (b)

In [31]:
N = 516083
p = factorLenstra(N)
q = N // p
print(N, "=", p, "*", q)
N == p*q

516083 = 569 * 907


True

In [32]:
N = 1833779393
p = factorLenstra(N)
q = N // p
print(N, "=", p, "*", q)
N == p*q

1833779393 = 12569 * 145897


True

In [33]:
N = 13487843290022369713
p = factorLenstra(N, 1200)
q = N // p
print(N, "=", p, "*", q)
N == p*q

13487843290022369713 = 1 * 13487843290022369713


True

### Exercise 6

In [35]:
# How to encode a message to a point
def messageToPoint(m, E, p, K = 100) :
    #K is the error tollerance
    # Message m needs to satisfy (m +1)K<p
    if (m +1) *K >= p:
        print (" Error tolerance or size of message space need to be changed !")
        return None
    for j in range (0, K - 1):
        x = K * m + j
        z = (x ** 3 + E [0] * x + E [1]) % p
        # print (x,z)
        if fast2Power (z, (p - 1) // 2, p) == 1:
            y = fast2Power (z ,(p + 1) // 4 , p)
            M = [x ,y]
            # print (P)
            return M
    print ("No encoding found , increase K")
    return None

#### (a)

In [314]:
def mSign(x, p):
    if x % p > (p - 1) // 2:
        return 1
    else:
        return 0

def yCoordinate(x, b, E, p):
    A, B = E
    a = (x**3 + A*x + B) % p
    if b == 0:
        return fast2Power(a, (p+1)//4, p) #Euler's Criterion
    else:
        return fast2Power(-a, (p+1)//4, p)
        
def elgamalEllipticEncrypt(P, QA, E, p, m):
    M = messageToPoint(m, E, p)
    k = randint(1, p - 1)
    kQA = doubleAndAdd(QA, k, E, p)
    C1, C2 = doubleAndAdd(P, k, E, p), addPoints(M, kQA, E, p)
    C1[1], C2[1] = mSign(C1[1], p), mSign(C2[1], p)
    return [C1, C2]


def elgamalEllipticDecrypt(C, nA, E, p, K = 100):
    C1, C2 = C
    x1, b1 = C1
    x2, b2 = C2 
    y1, y2 = yCoordinate(x1, b1, E, p), yCoordinate(x2, b2, E, p)
    negC1 = [x1, (p - y1) % p] #-C1
    nTimesNegC1 = doubleAndAdd(negC1, nA, E, p) #-nC1
    Mx, My = addPoints(C2, nTimesNegC1, E, p)
    m = Mx // K
    return m

### Exercise 7

In [316]:
text = "Hi there!"
m = textToInt(text)
print("plain text is:", m)
p = 2**256 - 2**32 - 977
E = [0,7]
M = messageToPoint(m, E, p)
print("original message is:", M)
n = 5
QA = doubleAndAdd(P, n, E, p)
C = elgamalEllipticEncrypt(P, QA, E, p, m)
print("encrypted message is:",C)
m2 = elgamalEllipticDecrypt(C, n, E, p)
print("Decrypted message is",M2)
t2 = intToText(m2)
print("Recovered message is:", t2)

plain text is: 616052571076890224968
original message is: [61605257107689022496803, 8295398745078820017989317430445340453279253918652070544627689389775750364059]
encrypted message is: [[58917061959581148227423278807723439527169794606298062532691033572706485637667, 0], [81955098761263548591028537998273832785362927196065101618440309128694332544632, 1]]
Decrypted message is 616052571076890224968
ÖßÎFI2ypOO¨|X¼ßÒÜÆwë>a¸Z


In [271]:
C1 = [81114305009337805755222806405877846608245959868065780326984929854162937583691, 1]
#[81114305009337805755222806405877846608245959868065780326984929854162937583691,
#90884524316682642771322773275895335659739945954679757806510641442186129463701]
C2 = [35081250314882291277328628031255744912038522158061460064453693760628471654504, 1]

In [293]:
x, y = C1
A, B = E
y = yCoordinate(x, 0, E, p)
P = [x, y]
print(P)
pointOnCurve(C1, E, p)

[81114305009337805755222806405877846608245959868065780326984929854162937583691, 24907564920633552652248211732792572193530038710960806232946942565722705207962]


False

In [297]:
C1 = [81114305009337805755222806405877846608245959868065780326984929854162937583691,
      90884524316682642771322773275895335659739945954679757806510641442186129463701]
pointOnCurve(C1, E, p)

True